In [0]:
%sql

SELECT * FROM silver.crm_cust_info LIMIT 100;

In [0]:
%sql
SELECT * FROM silver.crm_prd_info LIMIT 100;

In [0]:
%sql
SELECT * FROM silver.crm_sales_details LIMIT 100;

In [0]:
%sql
SELECT * FROM silver.erp_cust_az12 LIMIT 100;

In [0]:
%sql
SELECT * FROM silver.erp_loc_a101 LIMIT 100;

In [0]:
%sql
SELECT * FROM silver.erp_px_cat_g1v2 LIMIT 100;

# Integrating dim_customers

In [0]:
%sql
SELECT
    ROW_NUMBER() OVER (ORDER BY ci.cst_id ASC) AS customer_key,
    ci.cst_id AS costumer_id,
    ci.cst_key AS custmer_number,
    ci.cst_firstname AS first_name,
    ci.cst_lastname AS last_name,
    ci.cst_marital_status AS marital_status,
    CASE
        WHEN ci.cst_gndr IN ('Male', 'Female') THEN ci.cst_gndr
        WHEN ci.cst_gndr = 'N/A' AND ca.GEN IS NOT NULL AND ca.GEN != "N/A" THEN ca.GEN
        ELSE ci.cst_gndr
    END AS gender,
    ca.BDATE AS birth_date,
    la.CNTRY AS country
FROM silver.crm_cust_info as ci
LEFT JOIN silver.erp_cust_az12 as ca
ON ci.cst_key = ca.CID
LEFT JOIN silver.erp_loc_a101 as la
ON ci.cst_key = la.CID


# Integrating dim_product

In [0]:
%sql
SELECT
  ROW_NUMBER() OVER (ORDER BY pi.prd_id ASC) AS product_key,
  pi.prd_id AS product_id,
  pi.prd_key AS product_number,
  pi.prd_nm AS product_name,
  pi.prd_cat_id AS category_id,
  pc.CAT AS category,
  pc.SUBCAT AS subcategory,
  pc.MAINTENANCE AS maintenance_flag,
  pi.prd_cost AS product_cost,
  pi.prd_line AS prouduct_line,
  pi.prd_start_dt AS start_date
FROM silver.crm_prd_info AS pi
LEFT JOIN silver.erp_px_cat_g1v2 AS pc
ON pi.prd_cat_id = pc.ID 
LIMIT 200

# Integrating fact.sales

In [0]:
%sql
SELECT
	csd.sls_ord_num AS order_number,
	dc.customer_key AS customer_key,
	dp.product_key AS product_key,
	csd.sls_order_dt AS order_date,
	csd.sls_ship_dt AS ship_date,
	csd.sls_due_dt AS due_date,
	csd.sls_sales AS sales,
	csd.sls_quantity AS quantity,
	csd.sls_price AS price
FROM silver.crm_sales_details AS csd
LEFT JOIN gold.dim_products AS dp
ON csd.sls_prd_key = dp.product_number
LEFT JOIN gold.dim_customers AS dc
ON csd.sls_cust_id = dc.customer_id